# Notebook 00: Data Exploration

Orient yourself to the Sentinel-2-like synthetic scenes before we introduce the agent.

**What you'll learn:** What the band data looks like, how NDVI is computed from scratch, and why the ambiguity in Scene B motivates an adaptive pipeline.

In [ ]:
import sys, os
from pathlib import Path
# Move to project root so relative data paths (data/scenes/) resolve correctly
_here = Path(os.getcwd())
if not (_here / 'data' / 'scenes').exists() and (_here.parent / 'data' / 'scenes').exists():
    os.chdir(_here.parent)
sys.path.insert(0, str(Path(os.getcwd())))

import numpy as np
import matplotlib.pyplot as plt
from agent.scene import load_scene, get_band, get_metadata
from agent.types import BoundingBox

load_scene('scene_a')
meta = get_metadata()
print(f"Scene: {meta['scene_id']} | Date: {meta['date']} | "
      f"Shape: {meta['shape']} | Bands: {meta['bands']}")

## Band Overview

| Band | Name | Wavelength | Notes |
|------|------|------------|-------|
| B02 | Blue | 490 nm | Atmospheric scattering baseline |
| B03 | Green | 560 nm | Used in NDWI |
| B04 | Red | 665 nm | Chlorophyll absorption |
| B08 | NIR | 842 nm | Strong vegetation reflection |
| B8A | Red-edge | 865 nm | Sensitive to chlorophyll content |
| B11 | SWIR | 1610 nm | Leaf water content |
| B12 | SWIR2 | 2190 nm | Soil and dry vegetation |

In [ ]:
# Plot all 7 bands as a grid
bands = ['B02', 'B03', 'B04', 'B08', 'B8A', 'B11', 'B12']
names = ['Blue', 'Green', 'Red', 'NIR', 'Red-edge', 'SWIR', 'SWIR2']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, (band, name) in enumerate(zip(bands, names)):
    arr = get_band(band)
    im = axes[i].imshow(arr, cmap='gray',
                        vmin=np.percentile(arr, 2),
                        vmax=np.percentile(arr, 98))
    axes[i].set_title(f'{band} ({name})')
    axes[i].axis('off')
    plt.colorbar(im, ax=axes[i], fraction=0.046)
axes[-1].axis('off')
plt.suptitle('Scene A — All Bands (gray scale)', fontsize=14)
plt.tight_layout()
plt.show()

## Computing NDVI by Hand

NDVI = (NIR − Red) / (NIR + Red)

- NIR high, Red low → healthy green vegetation → NDVI near 1
- NIR low, Red elevated → stressed or senescent → NDVI near 0 or negative

Let's compute it directly in numpy — no tools yet:

In [ ]:
nir = get_band('B08')
red = get_band('B04')
ndvi = (nir - red) / (nir + red + 1e-8)  # epsilon avoids division by zero

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(nir, cmap='gray'); axes[0].set_title('NIR (B08)'); axes[0].axis('off')
axes[1].imshow(red, cmap='gray'); axes[1].set_title('Red (B04)'); axes[1].axis('off')
im = axes[2].imshow(ndvi, cmap='RdYlGn', vmin=-0.2, vmax=1.0)
axes[2].set_title('NDVI'); axes[2].axis('off')
plt.colorbar(im, ax=axes[2])
plt.suptitle('NDVI = (NIR - Red) / (NIR + Red)', fontsize=13)
plt.tight_layout(); plt.show()

print(f'NW quadrant mean NDVI: {ndvi[:100, :100].mean():.3f}')
print(f'SE quadrant mean NDVI: {ndvi[100:, 100:].mean():.3f}')

## The Ambiguity Problem — Scene B

Scene A has a clear anomaly. Scene B is harder. A fixed pipeline ("always run NDVI → threshold → alert") would struggle here. This is why we need an adaptive agent.

In [ ]:
load_scene('scene_b')
nir_b = get_band('B08'); red_b = get_band('B04')
ndvi_b = (nir_b - red_b) / (nir_b + red_b + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im0 = axes[0].imshow(ndvi, cmap='RdYlGn', vmin=-0.2, vmax=1.0)
axes[0].set_title(f'Scene A NDVI (mean={ndvi.mean():.3f})')
axes[0].axis('off'); plt.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(ndvi_b, cmap='RdYlGn', vmin=-0.2, vmax=1.0)
axes[1].set_title(f'Scene B NDVI (mean={ndvi_b.mean():.3f})')
axes[1].axis('off'); plt.colorbar(im1, ax=axes[1])
plt.suptitle('Same threshold, different diagnostic paths', fontsize=13)
plt.tight_layout(); plt.show()

print('Scene A: clear spatial anomaly → investigate NW quadrant')
print('Scene B: mild uniform depression → is this phenological? early stress?')